In [1]:
#%pip install --no-cache-dir \
 # "sagemaker>=2.232,<3.0.0" \
 # "transformers<5" \
#  "datasets==2.19.0"\
#  "huggingface_hub==0.23.0" \
#  "fsspec==2024.6.1" \
#  --quiet


%pip install --no-cache-dir \
  "datasets>=3.0.0" \
  "huggingface_hub" \
  "sagemaker>=2.232,<3.0.0" \
  "transformers<5" \
  --force-reinstall --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
dash 2.18.1 requires dash-core-components==2.0.0, which is not installed.
dash 2.18.1 requires dash-html-components==2.0.0, which is not installed.
dash 2.18.1 requires dash-table==5.0.0, which is not installed.
jupyter-ai 2.31.7 requires faiss-cpu!=1.8.0.post0,<2.0.0,>=1.8.0, which is not installed.
aiobotocore 2.22.0 requires botocore<1.37.4,>=1.37.2, but you have botocore 1.42.71 which is incompatible.
amazon-sagemaker-jupyter-ai-q-developer 1.2.9 requires numpy<=2.0.1, but you h

### IMPORT 

In [1]:
import fsspec, datasets, huggingface_hub
print(f"fsspec:          {fsspec.__version__}")
print(f"datasets:        {datasets.__version__}")
print(f"huggingface_hub: {huggingface_hub.__version__}")

fsspec:          2026.2.0
datasets:        4.8.2
huggingface_hub: 0.36.2


In [2]:
import transformers
import sagemaker
import boto3
import botocore
import os
from random import randint
from itertools import chain
from functools import partial
from sagemaker.huggingface import HuggingFace

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


### LOAD DATASET

**This dataset was built specifically for fine-tuning LLMs into customer service chatbots — covering intents common across 20 industry verticals including retail, banking, healthcare, travel, insurance and more.**

Bitext Customer Support:
- 100% customer service focused 
- Real support intents and responses 

In [1]:
import transformers
from datasets import load_dataset
from random import randrange
from pprint import pprint
from random import randint
from itertools import chain
from functools import partial

In [2]:
from datasets import load_dataset

raw_dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    split="train",
)
raw_dataset = raw_dataset.select(range(15000))

split_dataset = raw_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Train examples: {len(train_dataset)}")
print(f"Eval examples:  {len(eval_dataset)}")


Train examples: 14250
Eval examples:  750


In [3]:
raw_dataset.features

{'flags': Value(dtype='string', id=None),
 'instruction': Value(dtype='string', id=None),
 'category': Value(dtype='string', id=None),
 'intent': Value(dtype='string', id=None),
 'response': Value(dtype='string', id=None)}

In [4]:
print(f"data_set size:, {len(raw_dataset)}")
pprint(raw_dataset[randrange(len(raw_dataset))])

data_set size:, 15000
{'category': 'CANCEL',
 'flags': 'BILMP',
 'instruction': 'where could I see the early termination penalties?',
 'intent': 'check_cancellation_fee',
 'response': 'Unquestionably! If you are looking to view the early termination '
             'penalties, I recommend logging in to your account and navigating '
             'to the "Termination Fees" section. There you will find all the '
             'relevant information regarding the penalties for terminating '
             'your contract early.'}


In [5]:
pprint(raw_dataset[randrange(len(raw_dataset))])

{'category': 'ORDER',
 'flags': 'BL',
 'instruction': 'help to cancel purchase {{Order Number}}',
 'intent': 'cancel_order',
 'response': 'I completely comprehend your need for assistance in canceling '
             'your purchase with the order number {{Order Number}}. Allow me '
             'to guide you through the process:\n'
             '\n'
             '1. **Access Your Account**: Begin by logging into your {{Online '
             'Company Portal Info}}.\n'
             '2. **Locate Your Order Section**: Once logged in, navigate to '
             "the '{{Online Order Interaction}}' or '{{Online Order "
             "Interaction}}' section.\n"
             '3. **Find Your Specific Purchase**: Look for the purchase '
             'associated with the order number {{Order Number}} and click on '
             'it.\n'
             '4. **Initiate Cancellation**: Within the purchase details, you '
             "will find the option to '{{Online Order Interaction}}'. Please "
        

### LOAD MODEL

In [ ]:
from huggingface_hub import login

# Paste your  huggiggface token here

In [7]:
from transformers import AutoTokenizer

##tokenizer.pad_token will become the eos.token, meaning short sentence will be padded by words that _cls or token words

model_id = "meta-llama/Meta-Llama-3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

In [10]:
print(f"Vocab size:    {tokenizer.vocab_size}")
print(f"Pad token:     {tokenizer.pad_token}")
print(f"EOS token:     {tokenizer.eos_token}")
print(f"BOS token:     {tokenizer.bos_token}")
print(f"Padding side:  {tokenizer.padding_side}")

Vocab size:    128000
Pad token:     <|end_of_text|>
EOS token:     <|end_of_text|>
BOS token:     <|begin_of_text|>
Padding side:  right


In [11]:

# Llama-3 official chat template (manually set because the BASE model
# — not -Instruct — ships without one)
tokenizer.chat_template = (
    "{% set loop_messages = messages %}"
    "{% for message in loop_messages %}"
    "{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'"
    "+ message['content'] | trim + '<|eot_id|>' %}"
    "{% if loop.first and messages[0]['role'] != 'system' %}"
    "{% set content = bos_token + '<|start_header_id|>system<|end_header_id|>\n\n"
    "You are a helpful assistant<|eot_id|>' + content %}"
    "{% else %}"
    "{% set content = bos_token + content %}"
    "{% endif %}"
    "{{ content }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}"
    "{% endif %}"
)
 
RESPONSE_HEADER = "<|start_header_id|>assistant<|end_header_id|>\n\n"

In [12]:
# Llama-3 official chat template (manually set because the BASE model
# — not -Instruct — ships without one)


def format_and_mask(sample):
    """
    Builds the full chat-formatted text AND the matching label mask
    in one step, so the prompt portion of every example is excluded
    from the loss before any packing happens.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful and empathetic customer service "
                "assistant. Always be polite, professional and "
                "solution-focused."
            ),
        },
        {"role": "user", "content": sample["instruction"]},
        {"role": "assistant", "content": sample["response"]},
    ]
 
    # Full text (prompt + response), turn closed with <|eot_id|>
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
 
    # Prompt-only text (everything up to and including the assistant
    # header, i.e. where generation would begin)
    prompt_only_messages = messages[:-1]  # drop the assistant turn
    prompt_text = tokenizer.apply_chat_template(
        prompt_only_messages, tokenize=False, add_generation_prompt=True
    )
 
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
 
    prompt_len = len(prompt_ids)
 
    labels = full_ids.copy()
    # Mask everything that is prompt (system + user + assistant header)
    for i in range(min(prompt_len, len(labels))):
        labels[i] = -100
 
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }

In [13]:
# =====================================================================
# TOKENIZE — run format_and_mask over BOTH train and eval, separately
# =====================================================================
# Not batched=True — format_and_mask needs one example at a time to
# compute each example's individual prompt/response boundary.
 
tokenized_train = train_dataset.map(
    format_and_mask,
    remove_columns=list(train_dataset.features)
)
 
tokenized_eval = eval_dataset.map(
    format_and_mask,
    remove_columns=list(eval_dataset.features)
)
 
print(tokenized_train[4])
print(tokenized_eval[0])

Map:   0%|          | 0/14250 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

{'input_ids': [128000, 128006, 9125, 128007, 271, 2675, 527, 264, 11190, 323, 36681, 5411, 6130, 2532, 18328, 13, 24119, 387, 48887, 11, 6721, 323, 6425, 52373, 13, 128009, 128000, 128006, 882, 128007, 271, 53757, 1520, 7231, 264, 4062, 1427, 520, 856, 25637, 674, 6549, 2421, 128009, 128000, 128006, 78191, 128007, 271, 40, 3077, 50087, 7018, 430, 499, 1205, 264, 4062, 1427, 520, 701, 25637, 449, 279, 1396, 674, 3052, 35774, 5742, 3500, 13, 6914, 757, 7945, 499, 449, 430, 13, 16910, 499, 4587, 3493, 757, 449, 1063, 5217, 2038, 1093, 279, 2457, 2134, 477, 904, 1023, 3649, 5552, 311, 420, 25637, 30, 1115, 690, 1520, 757, 25539, 433, 369, 499, 6288, 323, 30357, 13, 128009], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [14]:

from itertools import chain
from functools import partial

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

In [15]:
tokenized_train

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 14250
})

In [16]:
train_dataset

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 14250
})

In [17]:
train_dataset[4]

{'flags': 'BLQ',
 'instruction': 'want help giving a quick look at my invoice #12588',
 'category': 'INVOICE',
 'intent': 'check_invoice',
 'response': "I've grasped that you need a quick look at your invoice with the number #{{Invoice Number}}. Let me assist you with that. Could you please provide me with some additional information like the date range or any other details related to this invoice? This will help me locate it for you quickly and accurately."}

In [18]:
tokenized_train[4]

{'input_ids': [128000,
  128006,
  9125,
  128007,
  271,
  2675,
  527,
  264,
  11190,
  323,
  36681,
  5411,
  6130,
  2532,
  18328,
  13,
  24119,
  387,
  48887,
  11,
  6721,
  323,
  6425,
  52373,
  13,
  128009,
  128000,
  128006,
  882,
  128007,
  271,
  53757,
  1520,
  7231,
  264,
  4062,
  1427,
  520,
  856,
  25637,
  674,
  6549,
  2421,
  128009,
  128000,
  128006,
  78191,
  128007,
  271,
  40,
  3077,
  50087,
  7018,
  430,
  499,
  1205,
  264,
  4062,
  1427,
  520,
  701,
  25637,
  449,
  279,
  1396,
  674,
  3052,
  35774,
  5742,
  3500,
  13,
  6914,
  757,
  7945,
  499,
  449,
  430,
  13,
  16910,
  499,
  4587,
  3493,
  757,
  449,
  1063,
  5217,
  2038,
  1093,
  279,
  2457,
  2134,
  477,
  904,
  1023,
  3649,
  5552,
  311,
  420,
  25637,
  30,
  1115,
  690,
  1520,
  757,
  25539,
  433,
  369,
  499,
  6288,
  323,
  30357,
  13,
  128009],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,

In [19]:


###new update
remainder = {
    "input_ids": [],
    "attention_mask": [],
    "labels": [],   
}
 
 
def chunk(sample, chunk_length=2048):
    """
    Packs tokenized examples into fixed-length chunks, carrying
    input_ids, attention_mask, AND labels through together so they
    stay aligned after concatenation.
    
    Take a batch of tokenized samples, flatten them into one long token stream,
    prepend leftover tokens from the previous batch, and split into fixed-size chunks.

    Input sample looks like:
    {
        "input_ids": [[...], [...], [...]],
        "attention_mask": [[...], [...], [...]]
    }

    Output looks like:
    {
        "input_ids": [[2048 tokens], [2048 tokens], ...],
        "attention_mask": [[2048 values], [2048 values], ...],
        "labels": [[2048 tokens], [2048 tokens], ...]
    }
    """
    
    
    global remainder
 
    concatenated_examples = {
        k: list(chain(*sample[k])) for k in sample.keys()
    }
    # Add leftover tokens from previous batch to the front
    concatenated_examples = {
        k: remainder[k] + concatenated_examples[k]
        for k in concatenated_examples.keys()
    }
   # Total number of tokens in this merged stream
    batch_total_length = len(concatenated_examples["input_ids"])
     # Keep only the largest multiple of chunk_length
    # Example:
    # if total = 5000 and chunk_length = 2048
    # usable = 4096
 
    if batch_total_length >= chunk_length:
        batch_total_length = (batch_total_length // chunk_length) * chunk_length
     # Split into fixed-size chunks
    # Example:
    # [1..4096] -> two chunks of 2048
 
    result = {
        k: [
            t[i: i + chunk_length]
            for i in range(0, batch_total_length, chunk_length)
        ]
        for k, t in concatenated_examples.items()
    }

     # Save leftover tokens that did not fit into a full chunk
    remainder = {
        k: concatenated_examples[k][batch_total_length:]
        for k in concatenated_examples.keys()
    }
 
    # NOTE: we no longer do result["labels"] = result["input_ids"].copy()
    # here — labels already exist (with -100 masking) from Fix 3 and
    # are carried through the same packing logic as input_ids above.
 
    return result

In [21]:
# --- Pack TRAIN set ---
remainder = {"input_ids": [], "attention_mask": [], "labels": []}
lm_train = tokenized_train.map(
    partial(chunk, chunk_length=2048),
    batched=True
)
print(f"Train packed samples: {len(lm_train)}")
 
# --- Pack EVAL set (remainder reset so train's leftovers don't leak in) ---
remainder = {"input_ids": [], "attention_mask": [], "labels": []}
lm_eval = tokenized_eval.map(
    partial(chunk, chunk_length=2048),
    batched=True
)
print(f"Eval packed samples: {len(lm_eval)}")
 
print(type(lm_train))
print(lm_train)
print(lm_eval)

Train packed samples: 1216
Eval packed samples: 62
<class 'datasets.arrow_dataset.Dataset'>
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1216
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 62
})


### CREATE A BUCKET AND MOVE THE DATASET TO THE BUCKET 

In [22]:
import boto3
import botocore

bucket_name = "llm-dataset-2026-tutorial"
s3 = boto3.client("s3")
region = boto3.Session().region_name
print(f"Your region: {region}")
 
try:
    s3.head_bucket(Bucket=bucket_name)
    print(f"ℹ️  Bucket already exists: {bucket_name}")
except botocore.exceptions.ClientError as e:
    error_code = e.response["Error"]["Code"]
    if error_code == "404":
        if region == "us-east-1":
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region},
            )
        print(f"✅ Bucket created: {bucket_name}")
    else:
        raise

Your region: eu-central-1
✅ Bucket created: llm-dataset-2026-tutorial


In [23]:
print(type(lm_dataset))
print(lm_dataset)

<class 'datasets.arrow_dataset.Dataset'>
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1286
})


In [24]:


# --- Upload BOTH train and eval sets, to separate S3 prefixes ---
 
region = boto3.Session().region_name
s3 = boto3.client("s3", region_name=region)
 
training_input_path = f"s3://{bucket_name}/processed/llama/bitext/train"
eval_input_path = f"s3://{bucket_name}/processed/llama/bitext/eval"
 
storage_opts = {
    "key": boto3.Session().get_credentials().access_key,
    "secret": boto3.Session().get_credentials().secret_key,
    "token": boto3.Session().get_credentials().token,
    "client_kwargs": {"region_name": region}
}
 
lm_train.save_to_disk(training_input_path, storage_options=storage_opts)
#print(f"Train dataset uploaded to {training_input_path}")
 
lm_eval.save_to_disk(eval_input_path, storage_options=storage_opts)
#print(f"Eval dataset uploaded to {eval_input_path}")

print("Dataset uploaded to S3 successfully")

Saving the dataset (0/1 shards):   0%|          | 0/1286 [00:00<?, ? examples/s]

Dataset uploaded to S3 successfully


## CREATE TRAINING JOB

In [8]:
import sagemaker
import time
from sagemaker.huggingface import HuggingFace

job_name = f"meta-Llama-3-8B-qlora-{time.strftime('%Y-%m-%d-%H-%M-%S', time.localtime())}"
training_input_path = "s3://llm-dataset-2026-tutorial/processed/llama/bitext/train"

hyperparameters = {
    "model_id":                    "meta-llama/Meta-Llama-3-8B",
    "dataset_path":                "/opt/ml/input/data/training",
    "eval_dataset_path":           "/opt/ml/input/data/eval",   
    "epochs":                       2,
    "per_device_train_batch_size":  1,
    "lr":                           2e-4,
    "merge_weights":                True,
}


huggingface_estimator = HuggingFace(
    entry_point          = "run_llama.py",
    source_dir           = "scripts",
    instance_type        = "ml.g5.2xlarge",
    instance_count       = 1,
    base_job_name        = job_name,
    role                 = role,
    volume_size          = 60,
    transformers_version = "4.36.0",
    pytorch_version      = "2.1.0",
    py_version           = "py310",
    hyperparameters      = hyperparameters,
    max_run              = 7200,
    environment          = {
        "HUGGINGFACE_HUB_CACHE":  "/tmp/.cache",
        "HUGGING_FACE_HUB_TOKEN": os.environ["HF_TOKEN"],
    }
)

In [ ]:
data = {
    "training": training_input_path,
    "eval": eval_input_path,
}
huggingface_estimator.fit(data, wait=True)


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: meta-Llama-3-8B-qlora-2026-03-18-17-05--2026-03-18-17-05-43-256


2026-03-18 17:05:44 Starting - Starting the training job
2026-03-18 17:05:44 Pending - Training job waiting for capacity...
2026-03-18 17:06:05 Pending - Preparing the instances for training...
2026-03-18 17:06:46 Downloading - Downloading the training image.....................
2026-03-18 17:09:54 Training - Training image download completed. Training in progress.bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.Trip